In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [1]:
import sys

import torch

from spice import SpiceEstimator

sys.path.append("../../..")
from weinhardt2026.utils.benchmarking_gru import GRUModel, training
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, get_dataset, generate_behavior
import spice_castro2025, spice_castro2025_2

## Load dataset

In [2]:
path_data = 'data/eckstein2024.csv'
test_sessions = (2,)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test, info_dataset = get_dataset(path_data=path_data, test_sessions=test_sessions, verbose=True)

Shape of dataset: torch.Size([4158, 150, 1, 13])
Number of participants: 862
Number of actions in dataset: 4


In [ ]:
# from spice import SpiceDataset

# # keep only 100 timesteps
# dataset_train = SpiceDataset(dataset_train.xs[:, :100], dataset_train.ys[:, :100])

# # keep only 100 participants for rapid prototyping
# keep_participants = torch.arange(0, 50)

# def keep_subset(dataset, subset):
#     participant_ids = dataset.xs[:, 0, 0, -1]
#     mask = torch.isin(participant_ids, subset)
#     return SpiceDataset(dataset.xs[mask], dataset.ys[mask])

# dataset_train = keep_subset(dataset_train, keep_participants)
# dataset_test = keep_subset(dataset_test, keep_participants)    


## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients
path_spice = 'params/spice_castro2025_2.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        save_path_spice=path_spice,
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [3]:
path_spice = 'params/spice_castro2025_2_noens_noprune.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=spice_castro2025_2.SpiceModel,
        spice_config=spice_castro2025_2.CONFIG,
        n_actions=dataset_train.n_actions,
        n_participants=dataset_train.n_participants,
        
        epochs=1000,
        warmup_steps=200,
        learning_rate=0.01,
        ensemble_size=1,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,

        verbose=True,
        # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

 24%|████████▉                            | 243/1000 [19:54<1:02:01,  4.92s/it, L(Train)=0.4903638, L(Val,RNN)=0.5158772, L(Val,SINDy)=0.8586412, Conv=1.40e-03]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
SPICE Model (Coefficients: 37):
value_reward_env[t+1]        = -0.04 1 + 0.961 value_reward_env[t] + 0.001 reward[t] + -0.005 value_reward_env^2 + -0.013 value_reward_env*reward[t] + 0.005 reward[t]^2 
value_reward_chosen[t+1]     = -0.377 1 + 1.005 value_reward_chosen[t] + 0.064 reward_env + 0.223 reward[t] + -0.005 value_reward_chosen^2 + 0.02 value_reward_chosen*reward_env + -0.149 value_reward_chosen*reward[t] + -0.005 reward_env^2 + -0.122 reward_env*reward[t] + 0.453 reward[t]^2 
value_reward_not_chosen[t+1] = -0.008 1 + 0.985 value_reward_not_chosen[t] + 0.014 reward_env + -0.003 value_reward_not_chosen^2 + 0.002 value_reward_not_chosen*reward_env + 0.001 reward_

  6%|▌         | 55/1000 [00:18<06:09,  2.56it/s, loss=0.0244261, n_params=37.00+/-0.00]

In [4]:
estimator.load_spice(path_spice)
# estimator.aggregate_coefficients()

In [5]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_env[t+1]        = 0.235 1 + 0.371 value_reward_env[t] + -0.178 reward[t] + 0.03 value_reward_env^2 + -0.438 value_reward_env*reward[t] + -0.306 reward[t]^2 
value_reward_chosen[t+1]     = -0.246 1 + 0.858 value_reward_chosen[t] + -0.651 reward_env + 0.618 reward[t] + 0.037 value_reward_mean + -0.024 value_reward_chosen^2 + 0.043 value_reward_chosen*reward_env + -0.077 value_reward_chosen*reward[t] + 0.05 value_reward_chosen*value_reward_mean + -0.084 reward_env^2 + -0.042 reward_env*reward[t] + -0.657 reward_env*value_reward_mean + 0.716 reward[t]^2 + -0.123 reward[t]*value_reward_mean + -0.03 value_reward_mean^2 
value_reward_not_chosen[t+1] = -0.067 1 + 0.959 value_reward_not_chosen[t] + 0.068 reward_env + 0.042 value_reward_mean + -0.002 value_reward_not_chosen^2 + 0.053 value_reward_not_chosen*reward_env + 0.001 value_reward_not_chosen*value_reward_mean + 0.016 reward_env^2 + -0.003 reward_env*value_reward_mean + 0.003 value_reward

## Benchmarking

### Castro2025 benchmark model

In [3]:
path_spice = 'params/spice_castro2025.pkl'

# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=dataset_train.n_participants,
    n_actions=dataset_train.n_actions,
    batch_first=True,
    )

path_cfs = path_spice.replace('spice_', 'cfs_')

In [ ]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    loss_kwargs={'label_smoothing': 0.0},
)

torch.save(cfs.state_dict(), path_cfs)

Epoch 1/1000: L(Train): 0.6583927869796753; L(Test): 0.6491681933403015
Epoch 2/1000: L(Train): 0.6574205160140991; L(Test): 0.6484070420265198
Epoch 3/1000: L(Train): 0.6564786434173584; L(Test): 0.6476723551750183
Epoch 4/1000: L(Train): 0.6555653810501099; L(Test): 0.6469632983207703
Epoch 5/1000: L(Train): 0.6546801924705505; L(Test): 0.6462777853012085
Epoch 6/1000: L(Train): 0.6538210511207581; L(Test): 0.6456145644187927
Epoch 7/1000: L(Train): 0.6529873013496399; L(Test): 0.6449723243713379
Epoch 8/1000: L(Train): 0.6521780490875244; L(Test): 0.6443491578102112
Epoch 9/1000: L(Train): 0.6513909101486206; L(Test): 0.6437438130378723
Epoch 10/1000: L(Train): 0.650625467300415; L(Test): 0.6431549191474915
Epoch 11/1000: L(Train): 0.6498801112174988; L(Test): 0.6425809264183044
Epoch 12/1000: L(Train): 0.6491535902023315; L(Test): 0.642021656036377
Epoch 13/1000: L(Train): 0.6484441757202148; L(Test): 0.6414744853973389
Epoch 14/1000: L(Train): 0.6477522253990173; L(Test): 0.640939

In [7]:
cfs.load_state_dict(torch.load(path_cfs, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [6]:
gru = GRUModel(
    n_actions=dataset_train.n_actions, 
    additional_inputs=2, 
    dropout=0.1,
    hidden_size=32,
    )
path_gru = path_spice.replace('spice_', 'gru_')

In [12]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    loss_kwargs={'label_smoothing': 0.0},
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 0.4995410144329071; L(Test): 1.8238345384597778
Epoch 2/1000: L(Train): 1.8106528520584106; L(Test): 0.7546141743659973
Epoch 3/1000: L(Train): 0.7678055763244629; L(Test): 0.7457813620567322
Epoch 4/1000: L(Train): 0.7755215167999268; L(Test): 0.8805248141288757
Epoch 5/1000: L(Train): 0.9214949011802673; L(Test): 0.8561751842498779
Epoch 6/1000: L(Train): 0.8995146751403809; L(Test): 0.7348523736000061
Epoch 7/1000: L(Train): 0.7798618674278259; L(Test): 0.6502336263656616
Epoch 8/1000: L(Train): 0.6937047243118286; L(Test): 0.6488674283027649
Epoch 9/1000: L(Train): 0.6794036030769348; L(Test): 0.65349942445755
Epoch 10/1000: L(Train): 0.6788702011108398; L(Test): 0.6388862133026123
Epoch 11/1000: L(Train): 0.6636232733726501; L(Test): 0.6207823157310486
Epoch 12/1000: L(Train): 0.6460587978363037; L(Test): 0.6204745769500732
Epoch 13/1000: L(Train): 0.6426492929458618; L(Test): 0.6185701489448547
Epoch 14/1000: L(Train): 0.6363800168037415; L(Test): 0.613431

In [9]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [8]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals
from analysis_generative import analysis_generative_behavior

## General Analysis

In [15]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [ ]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

## Analysis Model Evaluation

In [14]:
analysis_model_evaluation(
    dataset=dataset_train,
    # spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.620472,0.158088,13.0,0.0,233184.984375,466395.96875,466540.250
GRU,0.627344,0.156140,6852.0,0.0,227803.765625,469311.53125,545363.625
SPICE-RNN,1.000000,0.000000,NaN,0.0,0.000000,0.00000,0.000
SPICE,1.000000,0.000000,NaN,0.0,0.000000,0.00000,0.000


In [13]:
analysis_model_evaluation(
    dataset=dataset_test,
    # spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.eval().to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...


,Trial Lik.,(std),n_parameters,(std),NLL,AIC,BIC
Benchmark,0.599315,0.160264,13.0,0.0,65452.593750,130931.187500,131058.046875
GRU,0.618730,0.157360,6852.0,0.0,61376.554688,136457.109375,203322.843750
SPICE-RNN,1.000000,0.000000,NaN,0.0,0.000000,0.000000,0.000000
SPICE,1.000000,0.000000,NaN,0.0,0.000000,0.000000,0.000000


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

## Generative Behavior Analysis

In [ ]:
estimator.eval()
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice.csv'
)

estimator.use_sindy(False)
generate_behavior(
    model=estimator,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_spice_rnn.csv'
)

gru.eval()
generate_behavior(
    model=gru,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_gru.csv'
)

generate_behavior(
    model=cfs,
    dataset=dataset_train,
    save_dataset='data/eckstein2024_cfs.csv'
)

In [ ]:
analysis_generative_behavior(
    path_data_real=path_data,
    path_data_gru='data/eckstein2024_gru.csv',
    path_data_benchmark='data/eckstein2024_cfs.csv',
    path_data_spice='data/eckstein2024_spice.csv',
    path_data_spice_rnn='data/eckstein2024_spice_rnn.csv',
    output_dir='results',
)